# Overview

This notebook enriches the clean source metadata with Albumentations-generated image pairs before DINOv3 embedding extraction. Existing manually augmented records are excluded. The original metadata and images are not modified.

## 1. Imports

In [1]:
import os

import albumentations as A
import cv2
import pandas as pd

from pyprojroot import here
from tqdm.auto import tqdm

/Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/.venv/lib/python3.12/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error timed out>
  data = fetch_version_info()


## 2. Global params

In [2]:
PROJECT_ROOT = here()

RAW_DATASET_FOLDER = "data/meta/raw/"
INTERIM_DATASET_FOLDER = "data/meta/interim/"
IMAGES_FOLDER = "data/images/"

TARGET_INPUT_FILE = "metadata.parquet"
TARGET_OUTPUT_FILE = "metadata_augmented"

SLUDGE_IMAGE_COLUMN = "sludge_image_path"
LBA_IMAGE_COLUMN = "lba_image_path"
AUGMENTATION_FLAG_COLUMN = "is_augmented"

SLUDGE_AUGMENTED_FOLDER = "images-sludge-augmented"
LBA_AUGMENTED_FOLDER = "images-lba-augmented"

AUGMENTATIONS_PER_RECORD = 5
RANDOM_SEED = 42

AUGMENTATION_PARAMS = {
    "crop_size": 720,
    "output_size": 1024,

    "horizontal_flip_probability": 0.5,
    "vertical_flip_probability": 0.5,
    "random_rotate_90_probability": 0.5,

    "affine_probability": 0.35,
    "shift_limit": 0.03,
    "scale_limit": 0.07,
    "rotate_limit": 12,

    "clahe_probability": 0.15,
    "clahe_clip_limit": 1.25,
    "clahe_tile_grid_size": (8, 8),

    "sharpen_probability": 0.10,
    "sharpen_alpha": (0.05, 0.15),
    "sharpen_lightness": (0.8, 1.1),

    "hsv_probability": 0.15,
    "hue_shift_limit": 3,
    "saturation_shift_limit": 8,
    "value_shift_limit": 8,
}

## 3. Classes / Functions declaration

In [3]:
def build_augmentation_pipeline(params: dict, seed: int) -> A.Compose:
    crop_size = params["crop_size"]
    output_size = params["output_size"]

    return A.Compose(
        [
            A.PadIfNeeded(
                min_height=crop_size,
                min_width=crop_size,
                border_mode=cv2.BORDER_REFLECT_101,
                p=1.0,
            ),
            A.RandomCrop(width=crop_size, height=crop_size, p=1.0),
            A.Resize(width=output_size, height=output_size),
            A.HorizontalFlip(p=params["horizontal_flip_probability"]),
            A.VerticalFlip(p=params["vertical_flip_probability"]),
            A.RandomRotate90(p=params["random_rotate_90_probability"]),
            A.Affine(
                translate_percent={
                    "x": (-params["shift_limit"], params["shift_limit"]),
                    "y": (-params["shift_limit"], params["shift_limit"]),
                },
                scale=(1 - params["scale_limit"], 1 + params["scale_limit"]),
                rotate=(-params["rotate_limit"], params["rotate_limit"]),
                border_mode=cv2.BORDER_CONSTANT,
                fill=0,
                p=params["affine_probability"],
            ),
            A.CLAHE(
                clip_limit=params["clahe_clip_limit"],
                tile_grid_size=params["clahe_tile_grid_size"],
                p=params["clahe_probability"],
            ),
            A.Sharpen(
                alpha=params["sharpen_alpha"],
                lightness=params["sharpen_lightness"],
                p=params["sharpen_probability"],
            ),
            A.HueSaturationValue(
                hue_shift_limit=params["hue_shift_limit"],
                sat_shift_limit=params["saturation_shift_limit"],
                val_shift_limit=params["value_shift_limit"],
                p=params["hsv_probability"],
            ),
        ],
        seed=seed,
    )


def validate_metadata(df: pd.DataFrame) -> None:
    required_columns = {
        SLUDGE_IMAGE_COLUMN,
        LBA_IMAGE_COLUMN,
        AUGMENTATION_FLAG_COLUMN,
    }
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Metadata is missing required columns: {sorted(missing_columns)}")
    if df[AUGMENTATION_FLAG_COLUMN].isna().any():
        raise ValueError(f"Column '{AUGMENTATION_FLAG_COLUMN}' contains missing values.")
    if AUGMENTATIONS_PER_RECORD <= 0:
        raise ValueError("AUGMENTATIONS_PER_RECORD must be greater than zero.")


def load_rgb_image(relative_path: str) -> tuple:
    image_path = os.path.join(PROJECT_ROOT, IMAGES_FOLDER, relative_path)
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Image does not exist or cannot be read: {image_path}")

    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), image_path


def build_augmented_relative_path(
    source_relative_path: str,
    output_folder: str,
    record_index: int,
    augmentation_index: int,
) -> str:
    source_name = os.path.basename(source_relative_path)
    source_stem, source_extension = os.path.splitext(source_name)
    extension = source_extension or ".png"
    output_name = (
        f"{source_stem}__row_{record_index:06d}"
        f"__alb_{augmentation_index:02d}{extension}"
    )
    return os.path.join(output_folder, output_name)


def save_rgb_image(image, relative_path: str) -> None:
    output_path = os.path.join(PROJECT_ROOT, IMAGES_FOLDER, relative_path)
    was_saved = cv2.imwrite(output_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    if not was_saved:
        raise OSError(f"Failed to save augmented image: {output_path}")


def assert_augmented_folder_is_not_empty(folder_name: str) -> None:
    folder_path = os.path.join(PROJECT_ROOT, IMAGES_FOLDER, folder_name)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Augmented image folder does not exist: {folder_path}")
    if not any(os.path.isfile(entry.path) for entry in os.scandir(folder_path)):
        raise ValueError(f"Augmented image folder is empty: {folder_path}")


def enrich_with_augmentations(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.loc[df[AUGMENTATION_FLAG_COLUMN].eq(False)].copy().reset_index(drop=True)
    if clean_df.empty:
        raise ValueError("No non-augmented source records were found.")

    sludge_pipeline = build_augmentation_pipeline(AUGMENTATION_PARAMS, RANDOM_SEED)
    lba_pipeline = build_augmentation_pipeline(AUGMENTATION_PARAMS, RANDOM_SEED + 1)
    augmented_records = []

    for record_index, row in tqdm(
        clean_df.iterrows(),
        total=len(clean_df),
        desc="Augmenting image pairs",
    ):
        sludge_image, _ = load_rgb_image(row[SLUDGE_IMAGE_COLUMN])
        lba_image, _ = load_rgb_image(row[LBA_IMAGE_COLUMN])

        for augmentation_index in range(1, AUGMENTATIONS_PER_RECORD + 1):
            sludge_augmented = sludge_pipeline(image=sludge_image)["image"]
            lba_augmented = lba_pipeline(image=lba_image)["image"]

            sludge_relative_path = build_augmented_relative_path(
                row[SLUDGE_IMAGE_COLUMN],
                SLUDGE_AUGMENTED_FOLDER,
                record_index,
                augmentation_index,
            )
            lba_relative_path = build_augmented_relative_path(
                row[LBA_IMAGE_COLUMN],
                LBA_AUGMENTED_FOLDER,
                record_index,
                augmentation_index,
            )

            save_rgb_image(sludge_augmented, sludge_relative_path)
            save_rgb_image(lba_augmented, lba_relative_path)

            augmented_row = row.copy()
            augmented_row[SLUDGE_IMAGE_COLUMN] = sludge_relative_path
            augmented_row[LBA_IMAGE_COLUMN] = lba_relative_path
            augmented_row[AUGMENTATION_FLAG_COLUMN] = True
            augmented_records.append(augmented_row)

    augmented_df = pd.DataFrame(augmented_records, columns=clean_df.columns)
    enriched_df = pd.concat([clean_df, augmented_df], ignore_index=True)

    expected_rows = len(clean_df) * (1 + AUGMENTATIONS_PER_RECORD)
    if len(enriched_df) != expected_rows:
        raise AssertionError(
            f"Unexpected enriched row count: expected {expected_rows}, got {len(enriched_df)}."
        )

    return enriched_df

## 4. Actually running code

In [4]:
input_path = os.path.join(PROJECT_ROOT, RAW_DATASET_FOLDER, TARGET_INPUT_FILE)
output_csv_path = os.path.join(
    PROJECT_ROOT, INTERIM_DATASET_FOLDER, f"{TARGET_OUTPUT_FILE}.csv"
)
output_parquet_path = os.path.join(
    PROJECT_ROOT, INTERIM_DATASET_FOLDER, f"{TARGET_OUTPUT_FILE}.parquet"
)
sludge_output_path = os.path.join(
    PROJECT_ROOT, IMAGES_FOLDER, SLUDGE_AUGMENTED_FOLDER
)
lba_output_path = os.path.join(
    PROJECT_ROOT, IMAGES_FOLDER, LBA_AUGMENTED_FOLDER
)

print(f"Loading metadata from: {input_path}")
source_df = pd.read_parquet(input_path)
validate_metadata(source_df)

manual_augmentation_count = int(source_df[AUGMENTATION_FLAG_COLUMN].sum())
clean_record_count = int(source_df[AUGMENTATION_FLAG_COLUMN].eq(False).sum())
print(f"Source records: {len(source_df)}.")
print(f"Existing augmented records excluded: {manual_augmentation_count}.")
print(f"Clean records selected: {clean_record_count}.")

os.makedirs(sludge_output_path, exist_ok=True)
os.makedirs(lba_output_path, exist_ok=True)
os.makedirs(os.path.dirname(output_parquet_path), exist_ok=True)

enriched_df = enrich_with_augmentations(source_df)

assert_augmented_folder_is_not_empty(SLUDGE_AUGMENTED_FOLDER)
assert_augmented_folder_is_not_empty(LBA_AUGMENTED_FOLDER)

generated_record_count = int(enriched_df[AUGMENTATION_FLAG_COLUMN].sum())
expected_generated_count = clean_record_count * AUGMENTATIONS_PER_RECORD
if generated_record_count != expected_generated_count:
    raise AssertionError(
        f"Expected {expected_generated_count} augmented records, "
        f"got {generated_record_count}."
    )

enriched_df.to_csv(output_csv_path, index=False)
enriched_df.to_parquet(output_parquet_path, index=False)

print(f"Generated augmented records: {generated_record_count}.")
print(f"Final enriched records: {len(enriched_df)}.")
print(f"Sludge augmentations: {sludge_output_path}")
print(f"LBA augmentations: {lba_output_path}")
print(f"CSV metadata saved to: {output_csv_path}")
print(f"Parquet metadata saved to: {output_parquet_path}")

Loading metadata from: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/data/meta/raw/metadata.parquet
Source records: 382.
Existing augmented records excluded: 0.
Clean records selected: 382.


Augmenting image pairs:   0%|          | 0/382 [00:00<?, ?it/s]

Generated augmented records: 1910.
Final enriched records: 2292.
Sludge augmentations: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/data/images/images-sludge-augmented
LBA augmentations: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/data/images/images-lba-augmented
CSV metadata saved to: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/data/meta/interim/metadata_augmented.csv
Parquet metadata saved to: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/data/meta/interim/metadata_augmented.parquet
